# Recursion & Dynamic Programming — Combined Masterclass

The sixth notebook in the DSA series. Recursion and DP are the two topics that **separate strong candidates from average ones** in FAANG interviews. Most DP problems start as a recursive idea; you just keep optimizing.

The progression you'll internalize for almost every DP problem is the same four-step ladder:

```
1. Recursion          (brute force, exponential)
       ↓ add cache
2. Memoization        (top-down, polynomial time, O(states) space)
       ↓ flip order
3. Tabulation         (bottom-up, same time, often clearer transitions)
       ↓ drop old rows
4. Space-optimized    (same time, O(1) or O(min(m,n)) space)
```

In an interview, walk the interviewer up this ladder explicitly. State the time/space complexity at each rung and **explain the optimization** that took you from one rung to the next. That narration is half the signal.

**This notebook covers two things in one file:**
- Sections 1-4: **Recursion as a standalone topic** — the classic problems where DP doesn't apply (Hanoi, Josephus, divide & conquer).
- Sections 5-18: **DP pattern families** — each with the recursion → memo → tab → space-optimized walk-through where appropriate.

Out of scope: backtracking (gets its own notebook — N-Queens, Sudoku, permutations-with-constraints), binary search, sorting (mostly), graph DP (covered in graphs notebook).

## How to read this notebook
1. **Sections 1-2** — the recursion mental model. If you can't draw a recursion tree on a whiteboard, fix that first.
2. **Sections 3-4** — pure-recursion classics. These come up in interviews on their own.
3. **Sections 5-6** — the bridge to DP. Fibonacci is the Rosetta Stone — work through all four versions until the conversion is automatic.
4. **Sections 7-16** — the pattern families. Each follows the same template: recurrence → memoization → tabulation → space optimization → practice problems.
5. **Section 17** — advanced patterns you should at least know exist (bitmask DP, DP on trees, digit DP).
6. **Section 18** — the synthesis. Pattern-recognition cheat sheet, complexity table, interview narration prep, common bugs.

## The 8 master DP patterns

Every DP problem you'll see in interviews fits one of these:

| # | Pattern | Canonical problem |
|---|---------|-------------------|
| 1 | Linear / 1D state | Climbing stairs, House Robber |
| 2 | Kadane / running max | Max subarray, max product subarray |
| 3 | 0/1 Knapsack | Subset sum, Partition equal subset, Target sum |
| 4 | Unbounded knapsack | Coin change, Rod cutting |
| 5 | LIS / patience sorting | Longest Increasing Subsequence, Russian dolls |
| 6 | 2-string / 2D | LCS, Edit Distance, Distinct subsequences |
| 7 | Grid | Unique paths, Min path sum |
| 8 | Interval / MCM | Matrix chain mult., Burst balloons, Palindrome partitioning II |

State machine DP (stock problems) is essentially pattern 1 with multiple states per index — treated separately because it's a common interview category.

## The 4 master recursion patterns

| # | Pattern | Canonical problem |
|---|---------|-------------------|
| 1 | Linear recursion (single recursive call) | Factorial, sum of N, linked list ops |
| 2 | Multiple recursion (two+ calls, exponential) | Fibonacci, all subsets, Hanoi |
| 3 | Divide & conquer (T(n) = aT(n/b) + f(n)) | Merge sort, quicksort, max subarray D&C |
| 4 | Tail recursion (compiler-optimizable, but Python doesn't) | Accumulator-passing factorial |

Mutual recursion (function A calls function B which calls A) exists but is rare in interviews — mostly seen in parsers.


## 1. Recursion foundations

**Definition.** A recursive function is one that calls itself with a smaller version of the same problem.

### The 3 ingredients of every correct recursive function

1. **Base case** — when the input is small enough that you return directly without recursing. Missing or wrong base case → infinite recursion → `RecursionError`.
2. **Recursive case** — express the answer in terms of one or more **smaller** subproblems.
3. **Progress** — every recursive call must move *strictly closer* to a base case. If you can construct any input where it doesn't, your recursion is wrong.

### The call stack
Every recursive call pushes a stack frame. Python's default recursion limit is **1000**. For deep recursion you either bump it (`sys.setrecursionlimit(10**6)`) or convert to iteration. The space cost of a "simple" recursive function is **O(depth of recursion)** — not zero. This shows up constantly in interview complexity analysis.

### How to think about recursion (without going insane)

Don't try to trace recursion in your head past depth 2. Instead, use **the leap of faith**:

> Assume your function works correctly for all smaller inputs. Now write the one line that combines those answers to solve the current input.

This is sometimes called the **inductive mindset**. It's the same mental move as proof by induction:
- Base case: answer for n=0 (or whatever the smallest case is).
- Inductive step: assume `f(n-1)` is correct, write `f(n)` using it.

If both pieces are correct, the recursion is correct. Period. Don't trace the stack.

### Complexity analysis: recurrence relations

Every recursive function has a recurrence relation `T(n) = ...`. Standard ones:

| Recurrence | Solves to | Example |
|------------|-----------|---------|
| `T(n) = T(n-1) + O(1)` | O(n) | Factorial, sum of N |
| `T(n) = T(n-1) + O(n)` | O(n²) | Naive permutations of array (without pruning) |
| `T(n) = 2T(n-1) + O(1)` | O(2ⁿ) | Tower of Hanoi, naive Fibonacci, all subsets |
| `T(n) = T(n/2) + O(1)` | O(log n) | Binary search |
| `T(n) = T(n/2) + O(n)` | O(n) | Search in BST (balanced average) |
| `T(n) = 2T(n/2) + O(1)` | O(n) | Tree traversal |
| `T(n) = 2T(n/2) + O(n)` | O(n log n) | Merge sort, quicksort (average) |
| `T(n) = T(n-1) + T(n-2) + O(1)` | O(φⁿ) ≈ O(1.618ⁿ) | Naive Fibonacci |

**Master theorem** (for `T(n) = aT(n/b) + O(n^d)`):
- If `d < log_b(a)` → T(n) = O(n^(log_b a))
- If `d = log_b(a)` → T(n) = O(n^d · log n)
- If `d > log_b(a)` → T(n) = O(n^d)

For merge sort: a=2, b=2, d=1. log_b(a) = 1 = d → O(n log n). ✓

### The 5 questions to ask before solving any recursive problem

1. **What's the smallest input?** That's your base case.
2. **What does the function return?** Be specific. Wrong return contract = the most common recursion bug.
3. **How does the current call depend on the smaller call(s)?** That's the recurrence.
4. **Do I need extra parameters?** (accumulators, indices, paths) Often yes — use a helper.
5. **What's the recurrence relation?** Write it down. It tells you the complexity.


## 2. Recursion patterns: the 6 archetypes

### Pattern A — Linear recursion (one recursive call)
The simplest form. Each call makes one recursive call. Tree degenerates to a chain.

```
f(5) → f(4) → f(3) → f(2) → f(1) → base
```

Recurrence is typically `T(n) = T(n-1) + O(work)`. Depth = n → O(n) call stack.

### Pattern B — Multiple recursion (two or more recursive calls)
Each call branches. Forms a tree of calls. Without memoization, often exponential.

```
       f(4)
      /    \
   f(3)    f(2)
   / \     / \
 f(2) f(1) f(1) f(0)
 ...
```

This is where DP usually saves you — those overlapping subproblems get cached.

### Pattern C — Divide & conquer
A specific form of multiple recursion where each subproblem is a **fraction** of the original (n/2, n/3, etc.), not n-1. Master theorem applies. Examples: merge sort, quicksort, binary search.

### Pattern D — Tail recursion
The recursive call is the **last** thing the function does. No work after the call returns. In languages with TCO (tail call optimization), this is compiled to a loop — O(1) stack. **Python does NOT do TCO.** Still good to recognize the form, because tail-recursive functions are trivially convertible to iteration.

```python
# Tail recursive — note the accumulator
def factorial_tail(n, acc=1):
    if n <= 1: return acc
    return factorial_tail(n-1, acc * n)  # last action

# Not tail recursive — multiplication happens after the call returns
def factorial_normal(n):
    if n <= 1: return 1
    return n * factorial_normal(n-1)
```

### Pattern E — Helper / accumulator pattern
When the recursive function needs extra state that the caller doesn't care about (running sum, current path, current index), wrap it in a helper. Your `tower_of_hanoi` + `tower_of_hanoi_helper` is exactly this pattern.

### Pattern F — Mutual recursion
Function A calls B, B calls A. Rare in interviews but shows up in:
- Parsers (expression grammars)
- Coroutine-style state machines
- The is-even / is-odd toy example

### The "head vs tail" decision

In linear recursion, ask: **do I do work BEFORE the recursive call, or AFTER it?**

```python
# Head (work first, then recurse) — processes from the front
def print_list_head(arr, i=0):
    if i == len(arr): return
    print(arr[i])
    print_list_head(arr, i+1)

# Tail-after (recurse first, then work) — processes from the back
def print_list_reverse(arr, i=0):
    if i == len(arr): return
    print_list_reverse(arr, i+1)
    print(arr[i])   # prints in reverse order
```

This single trick — moving one line above or below the recursive call — is how you naturally implement preorder vs postorder, build-up vs unwind, and forward vs reverse.


In [ ]:
# Setup
import sys
sys.setrecursionlimit(10**6)   # raise default 1000 limit; needed for deeper recursion demos

# Quick demo: head vs tail processing
def head_print(arr, i=0):
    if i == len(arr): return
    print(arr[i], end=' ')
    head_print(arr, i+1)

def tail_print(arr, i=0):
    if i == len(arr): return
    tail_print(arr, i+1)
    print(arr[i], end=' ')

print("Head (forward):  ", end=''); head_print([1,2,3,4]); print()
print("Tail (reverse):  ", end=''); tail_print([1,2,3,4]); print()


## 3. Pure-recursion classics (no DP possible or needed)

Some problems are pure recursion exercises — they have no overlapping subproblems, so DP doesn't help. Interviewers ask them to test:
- Whether you can write a recurrence
- Whether you can reason about complexity from the recurrence
- Whether you understand the call stack

This is also the section where the "inductive mindset" pays off most.


### 3.1 Sum of N natural numbers

**Recurrence.** `sum(n) = n + sum(n-1)`, `sum(0) = 0`.

This is the cleanest possible linear recursion. The famous trap: the closed-form `n*(n+1)/2` is O(1), so the recursion is intentionally suboptimal — but the interview point is to demonstrate clean recursive thinking.

**Complexity:** Time O(n), Space O(n) (call stack).


In [ ]:
def sum_n(n):
    if n <= 0: return 0          # base case (handles 0 and negative defensively)
    return n + sum_n(n - 1)      # leap of faith: assume sum_n(n-1) works

assert sum_n(0) == 0
assert sum_n(5) == 15
assert sum_n(100) == 5050
print("sum_n: OK")


### 3.2 Factorial

**Recurrence.** `fact(n) = n * fact(n-1)`, `fact(0) = 1`.

**Complexity:** Time O(n), Space O(n).

Note: Python has unbounded integers, so factorial doesn't overflow — but in C/Java this overflows past `n=12` (int) or `n=20` (long).


In [ ]:
def factorial(n):
    if n <= 1: return 1
    return n * factorial(n - 1)

# Tail-recursive version (uses accumulator — Python still O(n) stack since no TCO)
def factorial_tail(n, acc=1):
    if n <= 1: return acc
    return factorial_tail(n - 1, acc * n)

# Iterative — O(1) stack
def factorial_iter(n):
    result = 1
    for i in range(2, n+1):
        result *= i
    return result

assert factorial(5) == 120
assert factorial_tail(5) == 120
assert factorial_iter(5) == 120
print("factorial: OK")


### 3.3 Power: a^b in O(log b) — fast exponentiation

**Naive recurrence.** `pow(a,b) = a * pow(a,b-1)` → O(b).

**Fast recurrence.**
```
pow(a, b) = pow(a, b/2)² if b is even
          = a * pow(a, b/2)² · 1 if b is odd
```
Recurrence: T(b) = T(b/2) + O(1) → **O(log b)**.

This is the classic interview question that tests whether you can spot a divide-and-conquer optimization. The key trick is **computing the half result once** and squaring — NOT recursing twice (that would still be O(b)).

**Complexity:** Time O(log b), Space O(log b) (recursion depth).


In [ ]:
def fast_pow(a, b):
    if b == 0: return 1
    if b == 1: return a
    half = fast_pow(a, b // 2)        # compute ONCE, square it — this is the optimization
    if b % 2 == 0:
        return half * half
    else:
        return half * half * a

# THE TRAP: recursing twice instead of caching half — same time as naive
def fast_pow_wrong(a, b):
    if b == 0: return 1
    if b % 2 == 0:
        return fast_pow_wrong(a, b//2) * fast_pow_wrong(a, b//2)  # BUG: T(b) = 2T(b/2) + O(1) = O(b)
    return a * fast_pow_wrong(a, b-1)

assert fast_pow(2, 10) == 1024
assert fast_pow(3, 7) == 2187
assert fast_pow(5, 0) == 1
print("fast_pow: OK — O(log b) time")


### 3.4 Tower of Hanoi

**Setup.** Three pegs (`from`, `aux`, `to`). N disks of decreasing size start on `from`. Move all to `to`. Constraints: one disk at a time; never put a larger disk on a smaller one.

**The recursive insight (the leap of faith).** To move N disks from `from` → `to`:
1. **Move top N-1 disks from `from` → `aux`** (using `to` as helper). Magic: assume the recursion works.
2. **Move the bottom disk (disk N) from `from` → `to`**. Just one move.
3. **Move the N-1 disks from `aux` → `to`** (using `from` as helper). Magic again.

That's it. Three lines.

```
Before:        from: [N,N-1,...,1]    aux: []           to: []
Step 1:        from: [N]              aux: [N-1,...,1]  to: []
Step 2:        from: []               aux: [N-1,...,1]  to: [N]
Step 3:        from: []               aux: []           to: [N,N-1,...,1]
```

**Recurrence.** `T(n) = 2T(n-1) + 1`, `T(1) = 1`. **Solves to T(n) = 2ⁿ - 1.**

**Complexity:** Time O(2ⁿ), Space O(n) (recursion depth, not move count — moves are output, not stored on the stack).

**Why this matters in interviews.** Hanoi is the textbook example of "you can't do better than exponential" — the **lower bound** on moves is 2ⁿ-1 (provable by induction on which disk moves when). So no DP, no clever trick will speed it up.


In [ ]:
def hanoi(n, src='A', aux='B', dst='C'):
    # Return list of (disk, from_peg, to_peg) moves.
    if n == 0: return []
    if n == 1: return [(1, src, dst)]
    moves = []
    moves += hanoi(n-1, src, dst, aux)         # step 1: move n-1 off the bottom
    moves.append((n, src, dst))                # step 2: move the bottom disk
    moves += hanoi(n-1, aux, src, dst)         # step 3: move n-1 onto target
    return moves

# Verify move count matches 2^n - 1
for n in [1, 2, 3, 4, 5, 10]:
    assert len(hanoi(n)) == 2**n - 1, f"n={n} failed"
print("Hanoi: OK")
print("n=3 sequence:", hanoi(3))


### 3.5 Josephus problem

**Setup.** N people in a circle (numbered 0 to N-1). Starting from position 0, count K people, kill the K-th. Continue from the next position. Find the position of the survivor.

**Brute force.** Simulate with a list — O(N·K) or O(N log N) with a deque. The reference file's helper version does this and is O(N·K) worst case.

**The elegant recurrence.** After the first kill (at position k-1), you have N-1 people left. The survivor's position in the *new* circle of N-1 people obeys the same recurrence. The trick: convert the survivor's position from the (N-1)-person subproblem back to the original numbering.

```
J(1) = 0
J(n) = (J(n-1) + k) % n
```

**Why the `+ k`?** After killing position `k-1`, the next person to start counting is at position `k`. So in the renumbered (N-1)-person circle, position `0` corresponds to original position `k`. The survivor's position shifts back by `k` (mod n).

**Complexity:** Time O(n), Space O(n) (stack). Can be flattened to O(1) space with iteration.


In [ ]:
def josephus(n, k):
    # Return 0-indexed position of the survivor.
    if n == 1: return 0
    return (josephus(n-1, k) + k) % n

# Iterative — O(1) space
def josephus_iter(n, k):
    survivor = 0
    for i in range(2, n+1):
        survivor = (survivor + k) % i
    return survivor

# Verify against your reference file's test cases
assert josephus(7, 3) == 3
assert josephus(8, 2) == 0
assert josephus(5, 3) == 3
assert josephus_iter(7, 3) == 3
assert josephus_iter(8, 2) == 0
print("Josephus: OK — recursive and iterative match")


### 3.6 String reversal via recursion

**Recurrence.** `reverse(s) = reverse(s[1:]) + s[0]`.

**Complexity:** Time **O(n²)** because slicing `s[1:]` is O(n) and we do it n times. Space O(n).

Better: pass indices and use the head-vs-tail trick.


In [ ]:
# Naive — O(n^2) due to slicing
def reverse_slow(s):
    if len(s) <= 1: return s
    return reverse_slow(s[1:]) + s[0]

# Better — O(n) time using indices
def reverse_fast(s):
    chars = list(s)
    def rec(lo, hi):
        if lo >= hi: return
        chars[lo], chars[hi] = chars[hi], chars[lo]
        rec(lo+1, hi-1)
    rec(0, len(chars)-1)
    return ''.join(chars)

assert reverse_slow("hello") == "olleh"
assert reverse_fast("hello") == "olleh"
print("String reversal: OK")


### 3.7 Generate all subsequences (subsets) of an array

**Recurrence.** At each index, two choices: include this element or skip it. → `T(n) = 2T(n-1) + O(work)` → O(2ⁿ · n) (n factor is for building each subset).

This is the **canonical "pick / not pick" recursion** that becomes 0/1 knapsack once you add a constraint. Memorize the template.

**Complexity:** Time O(2ⁿ · n), Space O(n) for stack + O(2ⁿ · n) for output.


In [ ]:
def all_subsequences(arr):
    out = []
    def rec(i, path):
        if i == len(arr):
            out.append(path[:])      # snapshot the current path
            return
        # Choice 1: skip arr[i]
        rec(i+1, path)
        # Choice 2: take arr[i]
        path.append(arr[i])
        rec(i+1, path)
        path.pop()                   # backtrack — undo the choice
    rec(0, [])
    return out

result = all_subsequences([1, 2, 3])
print("Subsequences of [1,2,3]:", result)
assert len(result) == 8  # 2^3


### 3.8 Generate all permutations

**Recurrence.** At index `i`, swap each remaining element into position `i` and recurse on `i+1`. → `T(n) = n · T(n-1) + O(n)` → **O(n! · n)**.

**Why n! · n?** There are n! permutations, and copying each takes O(n).


In [ ]:
def permutations(arr):
    out = []
    def rec(i):
        if i == len(arr):
            out.append(arr[:])
            return
        for j in range(i, len(arr)):
            arr[i], arr[j] = arr[j], arr[i]   # swap j into position i
            rec(i + 1)
            arr[i], arr[j] = arr[j], arr[i]   # swap back (backtrack)
    rec(0)
    return out

p = permutations([1, 2, 3])
print(f"{len(p)} permutations:", p)
assert len(p) == 6  # 3!


### Recursion practice problems

| Concept | LeetCode |
|---------|----------|
| Power of two/three via recursion | LC 231, LC 326 |
| Reverse linked list (recursive) | LC 206 |
| Merge two sorted lists (recursive) | LC 21 |
| Pow(x, n) — fast exponentiation | LC 50 |
| Tower of Hanoi | GFG classic |
| Josephus problem | GFG classic |
| Generate subsets | LC 78 |
| Permutations | LC 46 |
| String reversal recursive | warm-up |
| K-th symbol in grammar | LC 779 (sneaky recurrence) |


## 4. Divide & Conquer

**The D&C template:**
1. **Divide** the problem into smaller subproblems of the same kind.
2. **Conquer** each subproblem recursively.
3. **Combine** the subproblem solutions.

The recurrence form is `T(n) = a · T(n/b) + f(n)` where:
- `a` = number of subproblems
- `b` = the factor by which input size shrinks
- `f(n)` = cost of dividing + combining

Apply master theorem to get the closed form.

### 4.1 Merge sort

`T(n) = 2 T(n/2) + O(n)` — divides into 2 halves, merge is O(n). Master theorem: a=2, b=2, d=1; log_b(a)=1=d → **O(n log n)**.

**Space:** O(n) for the merge buffer + O(log n) for stack = **O(n)**. Not in-place.

**Stability:** Stable (because we take from the left half on ties).


In [ ]:
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return _merge(left, right)

def _merge(a, b):
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:        # <= keeps it stable (left wins on ties)
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])
    out.extend(b[j:])
    return out

assert merge_sort([5,2,8,1,9,3,7,4,6]) == [1,2,3,4,5,6,7,8,9]
assert merge_sort([3,1,2,3,1,2]) == [1,1,2,2,3,3]
print("merge_sort: OK — O(n log n) time, O(n) space, stable")


### 4.2 Quicksort

`T(n) = T(k) + T(n-k-1) + O(n)`. 
- **Best/average:** partition splits evenly → T(n) = 2T(n/2) + O(n) → **O(n log n)**.
- **Worst:** already sorted with naive pivot → T(n) = T(n-1) + O(n) → **O(n²)**.

**Space:** O(log n) average, O(n) worst (recursion depth). In-place (no merge buffer).

**Stability:** Not stable.

**The pivot question is the interview hook.** Random pivot or median-of-three avoids worst case in practice.


In [ ]:
import random

def quicksort(arr, lo=0, hi=None):
    if hi is None: hi = len(arr) - 1
    if lo >= hi: return
    p = _partition(arr, lo, hi)
    quicksort(arr, lo, p - 1)
    quicksort(arr, p + 1, hi)

def _partition(arr, lo, hi):
    # Random pivot to avoid worst case on sorted input
    rand_idx = random.randint(lo, hi)
    arr[rand_idx], arr[hi] = arr[hi], arr[rand_idx]
    pivot = arr[hi]
    i = lo - 1                              # i tracks last "small" position
    for j in range(lo, hi):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    arr[i+1], arr[hi] = arr[hi], arr[i+1]
    return i + 1

a = [5,2,8,1,9,3,7,4,6]
quicksort(a)
assert a == [1,2,3,4,5,6,7,8,9]
print("quicksort: OK — O(n log n) avg, O(n^2) worst, in-place, unstable")


### 4.3 Maximum subarray sum — D&C version

The famous problem is solved more elegantly by **Kadane's algorithm** (covered in section 8) at O(n). The D&C version is interesting because it teaches the "crossing the midpoint" trick that shows up in segment trees and similar structures.

`T(n) = 2T(n/2) + O(n)` → **O(n log n)** — strictly worse than Kadane's O(n), but a great teaching example.

**The combine step:** the maximum subarray either (a) lies entirely in the left half, (b) lies entirely in the right half, or (c) **crosses the midpoint**. The crossing case is computed in O(n) by sweeping outward from `mid`.


In [ ]:
def max_subarray_dc(arr):
    def rec(lo, hi):
        if lo == hi: return arr[lo]
        mid = (lo + hi) // 2
        left  = rec(lo, mid)                # case (a)
        right = rec(mid+1, hi)              # case (b)
        cross = _max_crossing(arr, lo, mid, hi)  # case (c)
        return max(left, right, cross)
    return rec(0, len(arr)-1)

def _max_crossing(arr, lo, mid, hi):
    # Best sum extending leftward from mid
    s, best_left = 0, float('-inf')
    for i in range(mid, lo-1, -1):
        s += arr[i]
        best_left = max(best_left, s)
    # Best sum extending rightward from mid+1
    s, best_right = 0, float('-inf')
    for i in range(mid+1, hi+1):
        s += arr[i]
        best_right = max(best_right, s)
    return best_left + best_right

assert max_subarray_dc([-2,1,-3,4,-1,2,1,-5,4]) == 6   # [4,-1,2,1]
assert max_subarray_dc([5,4,-1,7,8]) == 23
print("max_subarray_dc: OK — O(n log n) — Kadane's O(n) is strictly better")


### Divide & Conquer practice problems

| Concept | LeetCode |
|---------|----------|
| Merge sort | LC 912 (sort an array) |
| Quicksort (kth largest variant) | LC 215 |
| Pow(x, n) | LC 50 |
| Maximum subarray | LC 53 (Kadane > D&C) |
| Median of two sorted arrays | LC 4 |
| Count of smaller numbers after self | LC 315 (merge sort + index tracking) |
| Reverse pairs | LC 493 |
| Search a 2D matrix II | LC 240 (D&C version exists) |


## 5. The bridge: when recursion needs DP

**The two conditions for DP to apply** (these are non-negotiable, interviewers will test this verbatim):

1. **Overlapping subproblems** — the recursion solves the same subproblem more than once.
2. **Optimal substructure** — the optimal answer to the full problem can be built from optimal answers to smaller subproblems.

If only (1) is present without (2), memoizing gives speed but the answer might still be wrong (you need a greedy/different formulation).

If only (2) is present without (1), divide & conquer suffices — no need for DP. (Merge sort has optimal substructure but no overlapping subproblems.)

### Why naive recursion is slow: the Fibonacci tree

```
                  fib(5)
              /          \
          fib(4)         fib(3)
         /     \         /    \
      fib(3)  fib(2)  fib(2) fib(1)
      /  \    / \      / \
   fib(2) fib(1) ...
```

`fib(3)` is computed **3 times**. `fib(2)` is computed **5 times**. For `fib(n)` you get roughly φⁿ calls where φ ≈ 1.618 (the golden ratio). With n=40 that's already a billion-plus calls.

**The fix:** every distinct subproblem only needs to be computed once. With memoization, we drop from O(φⁿ) → O(n).

### The recursion-to-DP ladder

For any DP problem, walk this ladder in interviews:

| Step | What you do | Time | Space |
|------|-------------|------|-------|
| 1. Pure recursion | Write the recurrence | exponential | O(depth) |
| 2. Memoization | Add `@cache` or a dict | O(states · work_per_state) | O(states + depth) |
| 3. Tabulation | Convert to bottom-up loops | same | O(states) |
| 4. Space-optimized | Drop unused old states | same | often O(1) or O(min(m,n)) |

**Memoization vs Tabulation tradeoffs:**
- Memoization: easier to write (matches the recurrence directly), only computes subproblems you actually need (good for sparse state spaces), recursion overhead, risk of `RecursionError`.
- Tabulation: no recursion overhead, easier to space-optimize, must determine iteration order, computes ALL subproblems (bad if you only need a few).

In an interview: **start with memoization** (you wrote the recursion, just add a cache), then offer tabulation if asked to optimize.


## 6. Fibonacci — the canonical recursion → DP walkthrough

Master this single example and you've internalized 80% of DP. Every other problem in this notebook follows the same four-step transformation.

The recurrence: `fib(n) = fib(n-1) + fib(n-2)`, `fib(0)=0, fib(1)=1`.


### Version 1: Naive recursion — O(φⁿ) time, O(n) space

Direct translation of the recurrence. Beautiful, correct, **exponentially slow.**


In [ ]:
def fib_naive(n):
    if n < 2: return n
    return fib_naive(n-1) + fib_naive(n-2)

import time
start = time.time()
assert fib_naive(20) == 6765
print(f"fib_naive(20) = 6765 in {time.time()-start:.4f}s")

# Don't try fib_naive(40) — it takes minutes.


### Version 2: Memoization (top-down) — O(n) time, O(n) space

We add a dictionary (or `@cache` decorator) so each `fib(k)` is computed once. The recursion tree collapses into a linear chain of distinct subproblems.

**Optimization explained:** we trade O(n) extra memory for an exponential time speedup. The number of distinct subproblems is `n` (we compute `fib(0)..fib(n)` once each). Each lookup is O(1). So total work is **O(n) time, O(n) space** (cache + recursion depth).


In [ ]:
from functools import cache

@cache                                         # Python's built-in LRU memoization
def fib_memo(n):
    if n < 2: return n
    return fib_memo(n-1) + fib_memo(n-2)

# Manual version (interviews like to see you write the dict explicitly)
def fib_memo_manual(n, memo=None):
    if memo is None: memo = {}
    if n < 2: return n
    if n in memo: return memo[n]
    memo[n] = fib_memo_manual(n-1, memo) + fib_memo_manual(n-2, memo)
    return memo[n]

start = time.time()
assert fib_memo(100) == 354224848179261915075
print(f"fib_memo(100) computed in {time.time()-start:.6f}s — O(n)")
assert fib_memo_manual(50) == 12586269025


### Version 3: Tabulation (bottom-up) — O(n) time, O(n) space

We **flip the direction**: instead of asking "what's fib(n)?" and recursing down to base cases, we **start from the base cases** and build up. We replace recursion + cache with an array filled left-to-right.

**Optimization explained:** same time complexity as memoization, but we eliminate recursion overhead (no function call stack, no hash lookups). For Python this is roughly 2-5x faster in practice.

The key skill: **figure out the iteration order** so that when you compute `dp[i]`, the dependencies (`dp[i-1]`, `dp[i-2]`) are already filled. For 1D problems with `i-1, i-2` dependency this is trivially left-to-right.


In [ ]:
def fib_tab(n):
    if n < 2: return n
    dp = [0] * (n+1)
    dp[1] = 1
    for i in range(2, n+1):
        dp[i] = dp[i-1] + dp[i-2]
    return dp[n]

assert fib_tab(50) == 12586269025
assert fib_tab(100) == 354224848179261915075
print("fib_tab: OK — O(n) time, O(n) space, no recursion overhead")


### Version 4: Space-optimized — O(n) time, **O(1) space**

The big insight: at any point during tabulation, we only need the **last two values** (`dp[i-1]` and `dp[i-2]`) to compute `dp[i]`. The rest of the array is dead weight.

**Optimization explained:** we collapse the O(n) array into 2 variables. Time stays O(n), space drops from O(n) → **O(1)**. This is the most common DP space optimization — **whenever your recurrence only looks back a fixed number of steps, you can compress the dp array to a few variables.**


In [ ]:
def fib_optimal(n):
    if n < 2: return n
    prev2, prev1 = 0, 1
    for _ in range(2, n+1):
        curr = prev1 + prev2
        prev2, prev1 = prev1, curr
    return prev1

assert fib_optimal(50) == 12586269025
assert fib_optimal(100) == 354224848179261915075
print("fib_optimal: OK — O(n) time, O(1) space")


### Bonus: Fibonacci in O(log n) — matrix exponentiation

For huge n, there's a trick using fast exponentiation on the matrix `[[1,1],[1,0]]`:

```
[F(n+1)]   [1 1]^n   [1]
[ F(n) ] = [1 0]   · [0]
```

Raise the matrix to the nth power using fast exponentiation (O(log n) multiplications) → fib(n) in **O(log n) time**. Mention this in interviews when asked "can you do better than O(n)?" It rarely comes up but it's a credibility marker.

### The Fibonacci ladder summary

| Version | Time | Space | Optimization made |
|---------|------|-------|-------------------|
| Naive | O(φⁿ) ≈ O(1.618ⁿ) | O(n) stack | — |
| Memoization | O(n) | O(n) cache + O(n) stack | Each subproblem solved once (exponential → linear) |
| Tabulation | O(n) | O(n) array | No recursion overhead, easier to space-optimize |
| Space-optimized | O(n) | **O(1)** | Only need last 2 values, drop the array |
| Matrix exponentiation | **O(log n)** | O(log n) | Fast exponentiation on transition matrix |

**This same progression applies to every DP problem.** Internalize it.


## 7. 1D linear DP — the "single index" pattern

The simplest DP family. State is a single integer (typically an array index). Recurrence looks back a small number of steps.

**Pattern recognition signal:** "ways to reach position i" or "best value ending at position i".

### 7.1 Climbing stairs (LC 70)

You can take 1 or 2 steps. Count distinct ways to reach step n.

**Recurrence.** `ways(n) = ways(n-1) + ways(n-2)`. (Last step was either 1-step from n-1, or 2-step from n-2.)

This is literally Fibonacci. Same four versions, same complexity progression.


In [ ]:
# Space-optimized version (skipping intermediate steps — pattern is identical to Fibonacci)
def climb_stairs(n):
    if n <= 2: return n
    prev2, prev1 = 1, 2
    for _ in range(3, n+1):
        curr = prev1 + prev2
        prev2, prev1 = prev1, curr
    return prev1

assert climb_stairs(1) == 1
assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(5) == 8
print("climb_stairs: OK — O(n) time, O(1) space")


### 7.2 Min cost climbing stairs (LC 746)

`cost[i]` is the cost of stepping on stair i. You can start from index 0 or 1. Find the minimum cost to reach beyond the last stair.

**Recurrence.** `dp[i] = cost[i] + min(dp[i-1], dp[i-2])`.

`dp[i]` = min cost to **leave from** stair i (i.e., including its cost). Answer = `min(dp[n-1], dp[n-2])` (we can step off from either of the last two).


In [ ]:
def min_cost_climbing_stairs(cost):
    n = len(cost)
    prev2, prev1 = cost[0], cost[1]
    for i in range(2, n):
        curr = cost[i] + min(prev1, prev2)
        prev2, prev1 = prev1, curr
    return min(prev1, prev2)

assert min_cost_climbing_stairs([10,15,20]) == 15
assert min_cost_climbing_stairs([1,100,1,1,1,100,1,1,100,1]) == 6
print("min_cost_climbing_stairs: OK — O(n) time, O(1) space")


### 7.3 House Robber (LC 198)

Rob houses in a row. Can't rob two adjacent. Maximize total.

**Recurrence.** `dp[i] = max(dp[i-1], dp[i-2] + nums[i])`. Two choices: skip house i (carry over dp[i-1]) or rob house i (collect nums[i] + dp[i-2]).

**Why dp[i-2] and not dp[i-1] when robbing?** Because if you take house i, you can't have taken house i-1 → safest predecessor is dp[i-2].


In [ ]:
def rob(nums):
    if not nums: return 0
    if len(nums) == 1: return nums[0]
    prev2, prev1 = nums[0], max(nums[0], nums[1])
    for i in range(2, len(nums)):
        curr = max(prev1, prev2 + nums[i])
        prev2, prev1 = prev1, curr
    return prev1

assert rob([1,2,3,1]) == 4         # rob houses 0 and 2
assert rob([2,7,9,3,1]) == 12      # rob houses 0, 2, 4
print("rob: OK — O(n) time, O(1) space")


### 7.4 House Robber II — circular (LC 213)

Same as above but houses are in a **circle**. House 0 and house n-1 are adjacent.

**Trick.** The optimal answer either uses house 0 or it doesn't:
- Case A: don't rob house 0 → solve linear House Robber on `nums[1:]`.
- Case B: don't rob house n-1 → solve linear House Robber on `nums[:-1]`.
- Answer = max(A, B).

**This trick — splitting into two non-circular subproblems — comes up often** (also in circular subarray sum and similar).


In [ ]:
def rob2(nums):
    def rob_line(arr):
        if not arr: return 0
        if len(arr) == 1: return arr[0]
        prev2, prev1 = arr[0], max(arr[0], arr[1])
        for i in range(2, len(arr)):
            prev2, prev1 = prev1, max(prev1, prev2 + arr[i])
        return prev1
    if len(nums) == 1: return nums[0]
    return max(rob_line(nums[1:]), rob_line(nums[:-1]))

assert rob2([2,3,2]) == 3
assert rob2([1,2,3,1]) == 4
print("rob2: OK")


### 7.5 Decode Ways (LC 91)

Given a digit string, count decodings (1→A, 2→B, ..., 26→Z).

**Recurrence.** For each position i:
- If `s[i]` is non-zero, can decode it alone → `dp[i] += dp[i-1]`.
- If `s[i-1:i+1]` is between 10 and 26, can decode together → `dp[i] += dp[i-2]`.

**The traps.**
1. Leading zero in any 2-digit group ("06" is not valid).
2. "0" alone is not decodable.
3. Empty string typically returns 1 (by convention, depending on framing).


In [ ]:
def num_decodings(s):
    if not s or s[0] == '0': return 0
    n = len(s)
    prev2, prev1 = 1, 1                    # dp[-1]=1 (empty), dp[0]=1 (single non-zero char)
    for i in range(1, n):
        curr = 0
        # Single-char decode
        if s[i] != '0':
            curr += prev1
        # Two-char decode
        two = int(s[i-1:i+1])
        if 10 <= two <= 26:
            curr += prev2
        if curr == 0: return 0             # nothing valid at this position
        prev2, prev1 = prev1, curr
    return prev1

assert num_decodings("12") == 2            # AB or L
assert num_decodings("226") == 3           # BZ, VF, BBF
assert num_decodings("06") == 0
assert num_decodings("10") == 1
print("num_decodings: OK")


### 1D linear DP practice problems

| Concept | LeetCode |
|---------|----------|
| Climbing Stairs | LC 70 |
| Min Cost Climbing Stairs | LC 746 |
| House Robber | LC 198 |
| House Robber II (circular) | LC 213 |
| Decode Ways | LC 91 |
| Fibonacci Number | LC 509 |
| N-th Tribonacci Number | LC 1137 |
| Delete and Earn (Robber variant) | LC 740 |
| Maximum Alternating Subsequence Sum | LC 1911 |


## 8. Kadane's & jump-game family — "running max" DP

These are 1D DP problems where the DP value depends on `dp[i-1]` only, so they appear as one-pass scans rather than explicit tables. The DP is hidden, but it's there.

### 8.1 Maximum subarray (Kadane's algorithm) — LC 53

**The DP insight.** Let `dp[i]` = max sum of a subarray **ending at index i**. Then:

```
dp[i] = max(nums[i], dp[i-1] + nums[i])
```

Either start a new subarray at i, or extend the previous one. Answer = max of all dp[i].

**Optimization to O(1) space.** Only need `dp[i-1]`, not the array.

**Time O(n), Space O(1).** This is strictly better than the O(n log n) D&C version in section 4.3.


In [ ]:
def max_subarray(nums):
    best_ending_here = nums[0]
    best_overall = nums[0]
    for x in nums[1:]:
        best_ending_here = max(x, best_ending_here + x)  # extend or restart
        best_overall = max(best_overall, best_ending_here)
    return best_overall

assert max_subarray([-2,1,-3,4,-1,2,1,-5,4]) == 6
assert max_subarray([1]) == 1
assert max_subarray([5,4,-1,7,8]) == 23
print("max_subarray (Kadane's): OK — O(n) time, O(1) space")


### 8.2 Maximum product subarray (LC 152)

**The trick.** Unlike sum, where `max + x` is always `≥ x` (so we only track one running max), product can flip sign:
- A big negative · a negative number = big positive.

So we track **both** the running max and running min ending at i. On each step, the new candidates for max are `nums[i]`, `running_max · nums[i]`, `running_min · nums[i]`.


In [ ]:
def max_product(nums):
    cur_max = cur_min = best = nums[0]
    for x in nums[1:]:
        if x < 0:
            cur_max, cur_min = cur_min, cur_max   # swap because multiplying by neg flips
        cur_max = max(x, cur_max * x)
        cur_min = min(x, cur_min * x)
        best = max(best, cur_max)
    return best

assert max_product([2,3,-2,4]) == 6
assert max_product([-2,0,-1]) == 0
assert max_product([-2,3,-4]) == 24
print("max_product: OK — O(n) time, O(1) space")


### 8.3 Jump Game (LC 55)

Each `nums[i]` is the **max** jump length from i. Can you reach the last index?

**Greedy/DP hybrid.** Maintain `farthest` = the rightmost index reachable so far. Sweep left-to-right; if `i > farthest`, fail. Else update `farthest = max(farthest, i + nums[i])`.

**Why this is DP in disguise.** `dp[i]` = is i reachable? It collapses to a single "frontier" variable.

**Time O(n), Space O(1).**


In [ ]:
def can_jump(nums):
    farthest = 0
    for i, v in enumerate(nums):
        if i > farthest: return False           # gap we can't cross
        farthest = max(farthest, i + v)
        if farthest >= len(nums) - 1: return True   # early exit
    return True

assert can_jump([2,3,1,1,4]) == True
assert can_jump([3,2,1,0,4]) == False
print("can_jump: OK")


### 8.4 Jump Game II — minimum jumps (LC 45)

Now we want the **minimum number of jumps** to reach the end. (Guaranteed reachable.)

**BFS-as-DP.** At each "level" (jump count), find the farthest reachable. When the current pointer passes the end of the current level, increment jumps. This is implicitly the BFS-shortest-path on a graph where i → all j ≤ i + nums[i].

**Time O(n), Space O(1).**


In [ ]:
def jump(nums):
    jumps = 0
    cur_end = 0          # end of current jump's reach
    farthest = 0         # best reach achievable with one more jump
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == cur_end:               # exhausted current level
            jumps += 1
            cur_end = farthest
            if cur_end >= len(nums)-1: break
    return jumps

assert jump([2,3,1,1,4]) == 2
assert jump([2,3,0,1,4]) == 2
print("jump: OK — O(n) time")


### Kadane / jump practice problems

| Concept | LeetCode |
|---------|----------|
| Maximum Subarray | LC 53 |
| Maximum Product Subarray | LC 152 |
| Maximum Sum Circular Subarray | LC 918 |
| Maximum Absolute Sum of Subarray | LC 1749 |
| Jump Game | LC 55 |
| Jump Game II | LC 45 |
| Best Time to Buy and Sell Stock | LC 121 |
| Best Sightseeing Pair | LC 1014 |


## 9. 0/1 Knapsack family — "pick or skip" pattern

**Pattern recognition signal.** Given items each with a weight/cost, choose a subset to optimize some metric subject to a capacity constraint. Each item used **at most once.**

State: `dp[i][w]` = best answer using first `i` items with capacity `w`.

### 9.1 The 0/1 Knapsack template

**Setup.** N items, item i has weight `wt[i]` and value `val[i]`. Knapsack capacity W. Maximize total value.

**Recurrence.**
```
dp[i][w] = dp[i-1][w]                                  # skip item i
         = max(dp[i-1][w], val[i] + dp[i-1][w-wt[i]])  # if wt[i] <= w, also consider taking
```

**Time O(N·W), Space O(N·W).** Note: pseudo-polynomial — W is a *value*, not the bit length, so for large W this becomes slow.

**Space optimization to O(W).** `dp[i]` only depends on `dp[i-1]`. We can use a single array — **but we must iterate `w` from right to left** when doing the in-place update, so that `dp[w-wt[i]]` still refers to the previous row's value.


In [ ]:
def knapsack_01(weights, values, W):
    n = len(weights)
    dp = [[0] * (W+1) for _ in range(n+1)]
    for i in range(1, n+1):
        wt, val = weights[i-1], values[i-1]
        for w in range(W+1):
            dp[i][w] = dp[i-1][w]                            # skip
            if wt <= w:
                dp[i][w] = max(dp[i][w], val + dp[i-1][w-wt])  # take
    return dp[n][W]

# Space-optimized to O(W) — iterate w from high to low!
def knapsack_01_optimized(weights, values, W):
    dp = [0] * (W+1)
    for i in range(len(weights)):
        wt, val = weights[i], values[i]
        for w in range(W, wt-1, -1):       # REVERSE order — crucial for 0/1
            dp[w] = max(dp[w], val + dp[w-wt])
    return dp[W]

weights = [1, 3, 4, 5]
values  = [1, 4, 5, 7]
W = 7
assert knapsack_01(weights, values, W) == 9      # take items with weights 3 and 4
assert knapsack_01_optimized(weights, values, W) == 9
print("knapsack_01: OK — O(NW) time, can be O(W) space")


**Why does the 1D iteration have to be reverse?**

Forward iteration would let an item be picked multiple times in one outer iteration (because `dp[w-wt]` would already be updated with the current item). Reverse keeps `dp[w-wt]` referring to the "previous row" (i.e., before this item was considered).

This is the single most asked follow-up question on knapsack. Memorize the explanation.

### 9.2 Subset sum (LC 416 / 698 variants)

Can a subset of the array sum to exactly `target`? This is 0/1 knapsack with `value = weight` and we ask whether we can hit exactly target.

**Recurrence.** `dp[i][s] = dp[i-1][s] or dp[i-1][s - nums[i]]`.

`dp[i][s]` = can the first i numbers sum to s? Boolean DP.

**Optimization.** Space O(target) by collapsing the row, again iterating sum reverse.


In [ ]:
def subset_sum(nums, target):
    dp = [False] * (target + 1)
    dp[0] = True                                  # empty subset sums to 0
    for x in nums:
        for s in range(target, x-1, -1):          # REVERSE — each item once
            dp[s] = dp[s] or dp[s-x]
    return dp[target]

assert subset_sum([1, 5, 11, 5], 11) == True
assert subset_sum([1, 2, 3, 5], 8) == True
assert subset_sum([1, 2, 5], 4) == False
print("subset_sum: OK — O(N·target) time, O(target) space")


### 9.3 Partition equal subset sum (LC 416)

Can the array be partitioned into two subsets with equal sums?

**Reduction to subset sum.** Total must be even (else impossible). Then ask: can a subset sum to `total // 2`?


In [ ]:
def can_partition(nums):
    total = sum(nums)
    if total % 2: return False
    return subset_sum(nums, total // 2)

assert can_partition([1, 5, 11, 5]) == True   # [1,5,5] and [11]
assert can_partition([1, 2, 3, 5]) == False
print("can_partition: OK")


### 9.4 Target sum (LC 494)

Assign `+` or `-` to each number; count assignments that sum to `target`.

**The reframing.** Let `P` be the set of numbers with `+`, `N` the set with `-`. Then `P - N = target` and `P + N = total`. So `P = (target + total) / 2`. Now count subsets that sum to P → 0/1 knapsack **counting** variant.

**Recurrence (counting).** `dp[s] += dp[s - nums[i]]` instead of `or`. Same iteration order rules apply.


In [ ]:
def find_target_sum_ways(nums, target):
    total = sum(nums)
    # Infeasible cases
    if abs(target) > total or (target + total) % 2: return 0
    P = (target + total) // 2
    dp = [0] * (P + 1)
    dp[0] = 1
    for x in nums:
        for s in range(P, x-1, -1):
            dp[s] += dp[s - x]
    return dp[P]

assert find_target_sum_ways([1,1,1,1,1], 3) == 5
assert find_target_sum_ways([1], 1) == 1
print("find_target_sum_ways: OK")


### 0/1 Knapsack family practice problems

| Concept | LeetCode |
|---------|----------|
| 0/1 Knapsack (classic) | GFG 0/1 Knapsack |
| Partition Equal Subset Sum | LC 416 |
| Target Sum | LC 494 |
| Last Stone Weight II | LC 1049 (knapsack reformulation) |
| Ones and Zeroes | LC 474 (2D knapsack — two capacities) |
| Partition to K Equal Sum Subsets | LC 698 (bitmask DP) |
| Tallest Billboard | LC 956 |


## 10. Unbounded knapsack family — "infinite supply" pattern

**Pattern recognition.** Same as 0/1 knapsack but each item can be used **unlimited times.**

**The single change vs 0/1 knapsack.** When you "take" an item, you don't move to `i-1` — you stay at `i` (so the item can be taken again).

**Recurrence.**
```
dp[i][w] = dp[i-1][w]                              # skip
         = max(dp[i-1][w], val[i] + dp[i][w-wt[i]])  # take (note: dp[i], not dp[i-1]!)
```

**Space optimization to O(W).** Same as 0/1 but with **forward iteration** of w (not reverse) — because we *want* the same item to be reusable in the same row.

> **Trap reminder:** 0/1 = iterate capacity **reverse**, unbounded = iterate capacity **forward**. This is the single most asked DP follow-up trap. Burn it in.

### 10.1 Coin change (LC 322) — fewest coins to make amount

Coins of given denominations (infinite supply). Find min coins to make `amount`; return -1 if impossible.

**Recurrence.** `dp[a] = 1 + min(dp[a - coin])` over all coins.

`dp[a]` = min coins to make amount a. Initialize to `inf`. `dp[0] = 0`.


In [ ]:
def coin_change(coins, amount):
    INF = float('inf')
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a-c] + 1)
    return dp[amount] if dp[amount] != INF else -1

assert coin_change([1,2,5], 11) == 3      # 5+5+1
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
print("coin_change: OK — O(amount · |coins|) time")


### 10.2 Coin change II (LC 518) — count combinations

How many distinct **combinations** (order doesn't matter) make `amount`?

**The subtle but critical iteration-order trick:**
```python
for coin in coins:           # OUTER: iterate coins
    for a in range(coin, amount+1):    # INNER: iterate amounts forward
        dp[a] += dp[a - coin]
```

If you swap the loops (iterate amounts outer, coins inner) you'll count **permutations** instead of combinations (e.g., 1+2 and 2+1 counted separately). 

**Why?** Putting coins outer means each coin is "introduced" once; we never go back to an earlier coin after moving on. So combinations like {1, 2} are counted once with a defined order of consideration.


In [ ]:
def change(amount, coins):
    dp = [0] * (amount + 1)
    dp[0] = 1
    for coin in coins:                          # coins OUTER → combinations
        for a in range(coin, amount + 1):       # amount inner, forward (unbounded)
            dp[a] += dp[a - coin]
    return dp[amount]

assert change(5, [1,2,5]) == 4
assert change(3, [2]) == 0
assert change(10, [10]) == 1
print("change: OK — coins outer for combinations")


### 10.3 Combination sum IV — permutations variant (LC 377)

Same setup as coin change II, but **order matters** (1+2 and 2+1 are different).

**Iteration flip:** put amount outer, coins inner.


In [ ]:
def combination_sum4(nums, target):
    dp = [0] * (target + 1)
    dp[0] = 1
    for a in range(1, target + 1):              # amount OUTER → permutations
        for x in nums:
            if x <= a:
                dp[a] += dp[a - x]
    return dp[target]

assert combination_sum4([1,2,3], 4) == 7        # 1111, 112, 121, 211, 13, 31, 22
assert combination_sum4([9], 3) == 0
print("combination_sum4: OK — amount outer for permutations")


### 10.4 Rod cutting

A rod of length n. Price array `price[i]` for a rod piece of length i (1-indexed). Maximize revenue.

**Reformulation.** Unbounded knapsack: items = lengths 1..n, weight = length, value = price.

**Time O(n²), Space O(n).**


In [ ]:
def rod_cutting(price, n):
    # price[i] = price of a rod piece of length i+1 (0-indexed).
    dp = [0] * (n + 1)
    for length in range(1, n+1):
        for cut in range(1, length+1):
            dp[length] = max(dp[length], price[cut-1] + dp[length-cut])
    return dp[n]

# price[0]=$1 for len 1, price[1]=$5 for len 2, ...
assert rod_cutting([1,5,8,9,10,17,17,20], 8) == 22   # 2+6 → 5+17
print("rod_cutting: OK — O(n^2) time, O(n) space")


### Unbounded knapsack practice problems

| Concept | LeetCode |
|---------|----------|
| Coin Change (min coins) | LC 322 |
| Coin Change II (combinations) | LC 518 |
| Combination Sum IV (permutations) | LC 377 |
| Perfect Squares | LC 279 |
| Word Break | LC 139 (unbounded variant) |
| Word Break II | LC 140 |
| Rod Cutting | GFG classic |
| Integer Break | LC 343 |


## 11. Longest Increasing Subsequence (LIS) family

**Pattern.** "Longest [strictly/non-]increasing/decreasing subsequence." `dp[i]` = length of LIS **ending at i.**

### 11.1 LIS — O(n²) DP version (LC 300)

**Recurrence.** `dp[i] = 1 + max(dp[j]) for all j < i with nums[j] < nums[i]`. Else `dp[i] = 1` (just nums[i] alone).

Answer = `max(dp)`. Not `dp[n-1]` — the LIS doesn't have to end at the last element.

**Time O(n²), Space O(n).**


In [ ]:
def length_of_LIS_n2(nums):
    if not nums: return 0
    n = len(nums)
    dp = [1] * n
    for i in range(1, n):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

assert length_of_LIS_n2([10,9,2,5,3,7,101,18]) == 4   # [2,3,7,101]
assert length_of_LIS_n2([0,1,0,3,2,3]) == 4
print("length_of_LIS_n2: OK — O(n^2) time, O(n) space")


### 11.2 LIS — O(n log n) version using patience sorting + binary search

**The trick.** Maintain a `tails` array where `tails[k]` = the smallest possible tail value of an increasing subsequence of length k+1.

**Why this works.** `tails` is always sorted (monotonically increasing). When processing `nums[i]`:
- If `nums[i]` is larger than all tails, append it (extends the LIS by 1).
- Else, binary search for the position to **replace**: find the smallest `tails[k] >= nums[i]` and overwrite it with `nums[i]`. This **doesn't extend** the LIS length but **lowers** the tail for that length, making it easier to extend later.

After processing, `len(tails)` is the LIS length.

**Time O(n log n), Space O(n).** The optimization from O(n²) → O(n log n) replaces the inner `max(dp[j])` linear scan with a binary search.

⚠️ **Critical caveat.** `tails` is **NOT** an actual LIS — it's just a length tracker. You can't read the LIS off of `tails` directly. To reconstruct the LIS, you need to track parent pointers.


In [ ]:
from bisect import bisect_left

def length_of_LIS_nlogn(nums):
    tails = []
    for x in nums:
        idx = bisect_left(tails, x)
        if idx == len(tails):
            tails.append(x)        # extend
        else:
            tails[idx] = x         # replace — lower the tail for length idx+1
    return len(tails)

assert length_of_LIS_nlogn([10,9,2,5,3,7,101,18]) == 4
assert length_of_LIS_nlogn([0,1,0,3,2,3]) == 4
print("length_of_LIS_nlogn: OK — O(n log n) time, O(n) space")

# Note: for STRICTLY increasing → bisect_left. For NON-decreasing → bisect_right.


**bisect_left vs bisect_right gotcha.**
- Strict increase (`<`): use `bisect_left`. Equal elements replace.
- Non-strict (`≤`): use `bisect_right`. Equal elements extend.

This is a guaranteed interview trap.

### 11.3 Longest Bitonic Subsequence

A bitonic sequence increases then decreases. Length = LIS ending at i (left-to-right) + LDS ending at i (right-to-left) - 1.

Run LIS twice (once forward, once reverse), combine pointwise.

**Time O(n²) or O(n log n).**


In [ ]:
def longest_bitonic(nums):
    if not nums: return 0
    n = len(nums)
    # LIS ending at i (left-to-right)
    lis = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                lis[i] = max(lis[i], lis[j] + 1)
    # LDS ending at i (right-to-left)
    lds = [1] * n
    for i in range(n-1, -1, -1):
        for j in range(n-1, i, -1):
            if nums[j] < nums[i]:
                lds[i] = max(lds[i], lds[j] + 1)
    return max(lis[i] + lds[i] - 1 for i in range(n))

assert longest_bitonic([1, 11, 2, 10, 4, 5, 2, 1]) == 6   # 1,2,10,4,2,1
print("longest_bitonic: OK")


### 11.4 Russian Doll Envelopes (LC 354)

Each envelope (w, h). Envelope A fits in B iff `wA < wB and hA < hB`. Find longest chain.

**The trick.** Sort by `w` ascending, with `h` **descending** on ties. Then run LIS on the `h` values.

**Why h descending on ties?** Two envelopes with equal w can't fit inside each other → ensuring h is descending guarantees that at most one of them will be picked by the LIS (since LIS needs strict increase).


In [ ]:
def max_envelopes(envelopes):
    envelopes.sort(key=lambda e: (e[0], -e[1]))    # w asc, h desc on tie
    heights = [h for _, h in envelopes]
    return length_of_LIS_nlogn(heights)

assert max_envelopes([[5,4],[6,4],[6,7],[2,3]]) == 3
print("max_envelopes: OK — O(n log n)")


### LIS family practice problems

| Concept | LeetCode |
|---------|----------|
| Longest Increasing Subsequence | LC 300 |
| Number of LIS | LC 673 |
| Russian Doll Envelopes | LC 354 |
| Longest String Chain | LC 1048 |
| Increasing Triplet Subsequence | LC 334 (O(n) special case) |
| Largest Divisible Subset | LC 368 (sorted then LIS variant) |
| Wiggle Subsequence | LC 376 |
| Minimum Operations to Make Array Increasing | LC 1827 |


## 12. 2-string DP — the LCS family

**Pattern recognition.** Two sequences (strings, arrays), and you want to compute some "optimal alignment" between them. State = `(i, j)` for prefixes ending at i, j.

### 12.1 Longest Common Subsequence (LCS) — LC 1143

A subsequence keeps order but isn't necessarily contiguous.

**Recurrence.**
```
dp[i][j] = dp[i-1][j-1] + 1                  if s1[i-1] == s2[j-1]
         = max(dp[i-1][j], dp[i][j-1])        otherwise
```

Where `dp[i][j]` = LCS of `s1[:i]` and `s2[:j]`. So `dp[i][j]` looks at the prefixes of length i and j.

**Time O(m·n), Space O(m·n).**

**Space optimization.** `dp[i][j]` depends only on `dp[i-1][...]` and `dp[i][j-1]`. Keep only 2 rows: **O(min(m, n))** space.


In [ ]:
def LCS(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

# Space-optimized — O(min(m,n))
def LCS_optimized(s1, s2):
    if len(s1) < len(s2): s1, s2 = s2, s1     # ensure s2 is shorter
    m, n = len(s1), len(s2)
    prev = [0] * (n+1)
    for i in range(1, m+1):
        curr = [0] * (n+1)
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                curr[j] = prev[j-1] + 1
            else:
                curr[j] = max(prev[j], curr[j-1])
        prev = curr
    return prev[n]

assert LCS("abcde", "ace") == 3
assert LCS("abc", "abc") == 3
assert LCS("abc", "def") == 0
assert LCS_optimized("abcde", "ace") == 3
print("LCS: OK — O(m·n) time, O(min(m,n)) space optimized")


### 12.2 Longest Common Substring (subtly different)

Substring = contiguous. The recurrence resets to 0 on mismatch:

```
dp[i][j] = dp[i-1][j-1] + 1   if match
         = 0                   otherwise
```

Answer = `max(dp)`, not `dp[m][n]`.

**Time O(m·n), Space O(m·n) or O(min(m,n)).**


In [ ]:
def longest_common_substring(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n+1) for _ in range(m+1)]
    best = 0
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                best = max(best, dp[i][j])
    return best

assert longest_common_substring("abcde", "ace") == 1    # 'a' or 'c' or 'e' alone
assert longest_common_substring("OldSite:GeeksforGeeks.org", "NewSite:GeeksQuiz.com") == 10
print("longest_common_substring: OK")


### 12.3 Edit Distance (Levenshtein) — LC 72

Min operations (insert, delete, substitute) to convert s1 → s2.

**Recurrence.**
```
dp[i][j] = dp[i-1][j-1]                                       if s1[i-1] == s2[j-1]
         = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])      otherwise
                    delete     insert     substitute
```

Base cases: `dp[i][0] = i` (delete i chars), `dp[0][j] = j` (insert j chars).

**Time O(m·n), Space O(m·n) or O(min(m,n)).**

**Common interview trap:** explain which operation corresponds to which neighbor:
- `dp[i-1][j]` → delete from s1 (move up). 
- `dp[i][j-1]` → insert into s1 (move left). 
- `dp[i-1][j-1]` → substitute (move diagonal). 

Don't mix these up.


In [ ]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],     # delete
                                    dp[i][j-1],    # insert
                                    dp[i-1][j-1])  # substitute
    return dp[m][n]

assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert edit_distance("", "abc") == 3
print("edit_distance: OK — O(m·n) time")


### 12.4 Distinct Subsequences (LC 115)

Count ways `t` appears as a subsequence of `s`.

**Recurrence.**
```
dp[i][j] = dp[i-1][j-1] + dp[i-1][j]    if s[i-1] == t[j-1]
         = dp[i-1][j]                    otherwise
```

`dp[i][j]` = number of subsequences of `s[:i]` equal to `t[:j]`. The two branches when matching: either use `s[i-1]` to match `t[j-1]` (→ `dp[i-1][j-1]`), or skip it (→ `dp[i-1][j]`).

Base case: `dp[i][0] = 1` for all i (empty t is a subsequence of any prefix exactly once).


In [ ]:
def num_distinct(s, t):
    m, n = len(s), len(t)
    if n > m: return 0
    dp = [[0] * (n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = 1
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s[i-1] == t[j-1]:
                dp[i][j] = dp[i-1][j-1] + dp[i-1][j]
            else:
                dp[i][j] = dp[i-1][j]
    return dp[m][n]

assert num_distinct("rabbbit", "rabbit") == 3
assert num_distinct("babgbag", "bag") == 5
print("num_distinct: OK")


### 12.5 Wildcard / Regex Matching (LC 44 / 10)

Wildcard match (`?` = any single char, `*` = any sequence): LC 44.
Regex match (`.` = any single char, `*` = zero or more of preceding): LC 10.

**The trap.** In regex, `*` modifies the **previous** character — so when handling `p[j-1] == '*'`, you're really considering "use zero copies" (`dp[i][j-2]`) or "use one more copy" (`dp[i-1][j]` if previous char matches).

These are notorious "I knew the pattern but messed up an edge case" problems. Practice both.


In [ ]:
def is_match_regex(s, p):
    # LC 10: . matches any char, * = zero or more of preceding char.
    m, n = len(s), len(p)
    dp = [[False]*(n+1) for _ in range(m+1)]
    dp[0][0] = True
    # Handle patterns like a*, a*b*, ... matching empty string
    for j in range(1, n+1):
        if p[j-1] == '*':
            dp[0][j] = dp[0][j-2]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if p[j-1] == '*':
                # Zero of preceding
                dp[i][j] = dp[i][j-2]
                # One or more of preceding (if it matches s[i-1])
                if p[j-2] == '.' or p[j-2] == s[i-1]:
                    dp[i][j] = dp[i][j] or dp[i-1][j]
            else:
                if p[j-1] == '.' or p[j-1] == s[i-1]:
                    dp[i][j] = dp[i-1][j-1]
    return dp[m][n]

assert is_match_regex("aa", "a") == False
assert is_match_regex("aa", "a*") == True
assert is_match_regex("ab", ".*") == True
assert is_match_regex("mississippi", "mis*is*p*.") == False
print("is_match_regex: OK")


### 2-string DP practice problems

| Concept | LeetCode |
|---------|----------|
| Longest Common Subsequence | LC 1143 |
| Longest Common Substring | GFG classic |
| Edit Distance | LC 72 |
| Distinct Subsequences | LC 115 |
| Regular Expression Matching | LC 10 |
| Wildcard Matching | LC 44 |
| Interleaving String | LC 97 |
| Shortest Common Supersequence | LC 1092 |
| Delete Operation for Two Strings | LC 583 |
| Minimum ASCII Delete Sum | LC 712 |


## 13. Palindrome DP

**Pattern.** Substring problems where the structure of a palindrome (`s[i] == s[j]` and the inner part is palindromic) drives the recurrence. State = `(i, j)`.

### 13.1 Longest Palindromic Subsequence (LC 516)

Subsequence (not contiguous) that is a palindrome.

**Trick:** LPS(s) = LCS(s, reverse(s)). A palindrome in s aligns with itself in reversed s.

**Direct DP recurrence:**
```
dp[i][j] = dp[i+1][j-1] + 2                   if s[i] == s[j]
         = max(dp[i+1][j], dp[i][j-1])         otherwise
```
Base: `dp[i][i] = 1`. Iterate by length, not by i.

**Time O(n²), Space O(n²) → O(n).**


In [ ]:
def longest_palindromic_subsequence(s):
    n = len(s)
    dp = [[0]*n for _ in range(n)]
    for i in range(n):
        dp[i][i] = 1
    # Fill by increasing length
    for length in range(2, n+1):
        for i in range(n - length + 1):
            j = i + length - 1
            if s[i] == s[j]:
                dp[i][j] = (dp[i+1][j-1] if length > 2 else 0) + 2
            else:
                dp[i][j] = max(dp[i+1][j], dp[i][j-1])
    return dp[0][n-1]

assert longest_palindromic_subsequence("bbbab") == 4   # bbbb
assert longest_palindromic_subsequence("cbbd") == 2
print("longest_palindromic_subsequence: OK")


### 13.2 Longest Palindromic Substring (LC 5)

Substring (contiguous). Two common approaches:

**(a) DP — O(n²) time, O(n²) space.** `dp[i][j] = True` iff `s[i..j]` is a palindrome.
```
dp[i][j] = (s[i] == s[j]) and (j-i < 2 or dp[i+1][j-1])
```

**(b) Expand around center — O(n²) time, O(1) space.** For each of the 2n-1 possible centers (n char-centers + n-1 gap-centers), expand outward while characters match.

(b) is the preferred interview answer because it's **strictly better in space**.

**(c) Manacher's algorithm — O(n) time.** Mention only if asked for sub-quadratic; rarely required.


In [ ]:
def longest_palindromic_substring(s):
    # Expand-around-center, O(n^2) time, O(1) space.
    if not s: return ""
    start, end = 0, 0
    def expand(l, r):
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1; r += 1
        return l+1, r-1
    for i in range(len(s)):
        l1, r1 = expand(i, i)        # odd-length center
        l2, r2 = expand(i, i+1)      # even-length center
        if r1 - l1 > end - start: start, end = l1, r1
        if r2 - l2 > end - start: start, end = l2, r2
    return s[start:end+1]

assert longest_palindromic_substring("babad") in ("bab", "aba")
assert longest_palindromic_substring("cbbd") == "bb"
print("longest_palindromic_substring: OK")


### 13.3 Palindrome Partitioning II (LC 132)

Min cuts to partition s into all-palindrome substrings.

**Recurrence.** Let `cuts[i]` = min cuts for prefix `s[:i+1]`.
```
cuts[i] = 0                          if s[:i+1] is a palindrome
        = min(cuts[j] + 1) for all j where s[j+1..i] is a palindrome
```

Precompute `is_palin[i][j]` in O(n²), then fill `cuts` in O(n²). Total **O(n²) time, O(n²) space**.


In [ ]:
def min_cut(s):
    n = len(s)
    is_palin = [[False]*n for _ in range(n)]
    for i in range(n): is_palin[i][i] = True
    for length in range(2, n+1):
        for i in range(n - length + 1):
            j = i + length - 1
            if s[i] == s[j] and (length == 2 or is_palin[i+1][j-1]):
                is_palin[i][j] = True
    cuts = [0] * n
    for i in range(n):
        if is_palin[0][i]:
            cuts[i] = 0
        else:
            cuts[i] = float('inf')
            for j in range(i):
                if is_palin[j+1][i]:
                    cuts[i] = min(cuts[i], cuts[j] + 1)
    return cuts[n-1]

assert min_cut("aab") == 1          # aa | b
assert min_cut("a") == 0
assert min_cut("ab") == 1
print("min_cut: OK — O(n^2) time")


### Palindrome DP practice problems

| Concept | LeetCode |
|---------|----------|
| Longest Palindromic Substring | LC 5 |
| Longest Palindromic Subsequence | LC 516 |
| Palindromic Substrings (count) | LC 647 |
| Palindrome Partitioning | LC 131 (backtracking, not DP) |
| Palindrome Partitioning II (min cuts) | LC 132 |
| Minimum Insertion Steps to Make a String Palindrome | LC 1312 |
| Valid Palindrome III (k deletions) | LC 1216 |


## 14. Grid DP

**Pattern.** 2D matrix. From each cell you can typically move right or down (or with obstacles). Find paths, min/max sums, etc.

**State.** `dp[i][j]` = answer for reaching cell (i, j). 

### 14.1 Unique Paths (LC 62)

m×n grid. Count paths from (0,0) to (m-1, n-1) moving only right or down.

**Recurrence.** `dp[i][j] = dp[i-1][j] + dp[i][j-1]`. Base: `dp[0][j] = dp[i][0] = 1`.

**Space optimization.** Only need previous row → O(n) space.

**Bonus formula.** Pure math: `C(m+n-2, m-1)`. Useful to mention.


In [ ]:
def unique_paths(m, n):
    dp = [1] * n
    for i in range(1, m):
        for j in range(1, n):
            dp[j] += dp[j-1]      # dp[j] (old) is dp[i-1][j]; dp[j-1] (new) is dp[i][j-1]
    return dp[-1]

assert unique_paths(3, 7) == 28
assert unique_paths(3, 3) == 6
print("unique_paths: OK — O(m·n) time, O(n) space")


### 14.2 Unique Paths II — with obstacles (LC 63)

Same setup but `1` means obstacle. If a cell is an obstacle, `dp[i][j] = 0`.

**The trap:** start cell (0,0) being an obstacle means 0 paths.


In [ ]:
def unique_paths_with_obstacles(grid):
    m, n = len(grid), len(grid[0])
    if grid[0][0] == 1 or grid[m-1][n-1] == 1: return 0
    dp = [0] * n
    dp[0] = 1
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                dp[j] = 0
            elif j > 0:
                dp[j] += dp[j-1]
    return dp[-1]

assert unique_paths_with_obstacles([[0,0,0],[0,1,0],[0,0,0]]) == 2
print("unique_paths_with_obstacles: OK")


### 14.3 Minimum Path Sum (LC 64)

m×n grid of non-negative values. Find min sum from (0,0) to (m-1, n-1).

**Recurrence.** `dp[i][j] = grid[i][j] + min(dp[i-1][j], dp[i][j-1])`.

**Space optimization:** in-place, O(1) extra.


In [ ]:
def min_path_sum(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0: continue
            elif i == 0: grid[i][j] += grid[i][j-1]
            elif j == 0: grid[i][j] += grid[i-1][j]
            else:        grid[i][j] += min(grid[i-1][j], grid[i][j-1])
    return grid[m-1][n-1]

assert min_path_sum([[1,3,1],[1,5,1],[4,2,1]]) == 7    # 1→3→1→1→1
print("min_path_sum: OK — O(m·n) time, O(1) extra space")


### 14.4 Maximal Square (LC 221) and Maximal Rectangle (LC 85)

**Maximal Square.** Largest all-1 square in a binary matrix.

**Recurrence.** `dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])` if `grid[i][j] == 1` else 0.

`dp[i][j]` = side length of the largest all-1 square with bottom-right at (i, j).


In [ ]:
def maximal_square(matrix):
    if not matrix: return 0
    m, n = len(matrix), len(matrix[0])
    dp = [[0]*(n+1) for _ in range(m+1)]
    best = 0
    for i in range(1, m+1):
        for j in range(1, n+1):
            if matrix[i-1][j-1] == "1":
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
                best = max(best, dp[i][j])
    return best * best

assert maximal_square([["1","0","1","0","0"],
                        ["1","0","1","1","1"],
                        ["1","1","1","1","1"],
                        ["1","0","0","1","0"]]) == 4
print("maximal_square: OK")


### 14.5 Dungeon Game (LC 174) — backward DP

A knight in the top-left has HP, must reach bottom-right alive. Cells can damage or heal. Find min starting HP.

**The trap.** Forward DP doesn't work — the choice of path depends on the future. You can't decide at (i,j) without knowing the min HP needed downstream.

**Trick: iterate backward.** `dp[i][j]` = min HP needed when arriving at (i,j) to survive to the end.
```
need = min(dp[i+1][j], dp[i][j+1]) - dungeon[i][j]
dp[i][j] = max(1, need)        # at least 1 HP required at all times
```

This is the canonical "fill the DP backward" example.


In [ ]:
def calculate_minimum_HP(dungeon):
    m, n = len(dungeon), len(dungeon[0])
    INF = float('inf')
    dp = [[INF]*(n+1) for _ in range(m+1)]
    dp[m][n-1] = dp[m-1][n] = 1
    for i in range(m-1, -1, -1):
        for j in range(n-1, -1, -1):
            need = min(dp[i+1][j], dp[i][j+1]) - dungeon[i][j]
            dp[i][j] = max(1, need)
    return dp[0][0]

assert calculate_minimum_HP([[-2,-3,3],[-5,-10,1],[10,30,-5]]) == 7
print("calculate_minimum_HP: OK — backward DP")


### Grid DP practice problems

| Concept | LeetCode |
|---------|----------|
| Unique Paths | LC 62 |
| Unique Paths II (obstacles) | LC 63 |
| Minimum Path Sum | LC 64 |
| Maximal Square | LC 221 |
| Maximal Rectangle | LC 85 |
| Dungeon Game (backward DP) | LC 174 |
| Cherry Pickup | LC 741 (3D DP — two paths) |
| Cherry Pickup II | LC 1463 |
| Out of Boundary Paths | LC 576 |
| Triangle | LC 120 |


## 15. Interval / Matrix Chain Multiplication DP

**Pattern.** "Choose a partition point on an interval and combine the left and right results." State = `(i, j)` for an interval. Iteration order: **by interval length, not by i or j directly**.

These problems are characterized by **trying every possible split inside an interval**:
```
dp[i][j] = best over k in [i+1, j-1] of f(dp[i][k], dp[k][j], cost_of_combining_at_k)
```

### 15.1 Matrix Chain Multiplication (MCM)

Given dimensions `p[]` where matrix i is `p[i-1] × p[i]`, find min scalar multiplications to compute the chain.

**Why DP?** Multiplication is associative — but the order of parenthesization changes the cost dramatically.

**Recurrence.**
```
dp[i][j] = min over k in [i, j-1] of  dp[i][k] + dp[k+1][j] + p[i-1]*p[k]*p[j]
```

`dp[i][j]` = min cost to multiply matrices i through j. Base: `dp[i][i] = 0`.

**Time O(n³), Space O(n²).** Classic. Iterate `length` outer, then `i`, then `k`.


In [ ]:
def matrix_chain_order(p):
    # p has length n+1; matrix i (1-indexed) is p[i-1] x p[i].
    n = len(p) - 1
    dp = [[0]*(n+1) for _ in range(n+1)]
    for length in range(2, n+1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = dp[i][k] + dp[k+1][j] + p[i-1]*p[k]*p[j]
                dp[i][j] = min(dp[i][j], cost)
    return dp[1][n]

# Matrices 10x30, 30x5, 5x60 → optimal cost = 4500
assert matrix_chain_order([10, 30, 5, 60]) == 4500
print("matrix_chain_order: OK — O(n^3) time")


### 15.2 Burst Balloons (LC 312)

Burst balloons one at a time; bursting balloon i gives you `nums[left] * nums[i] * nums[right]` (where left and right are the adjacent ones at that moment, with virtual 1s at the boundaries). Maximize total coins.

**The trick: pick the LAST balloon to burst, not the first.** If you pick the first to burst, the neighbors keep changing — hard to recurse on. If you pick the **last** balloon in the interval to burst, its neighbors are **fixed** (the boundary 1s or the outer balloons of the interval).

```
dp[i][j] = max over k in [i, j] of  nums[i-1]*nums[k]*nums[j+1] + dp[i][k-1] + dp[k+1][j]
```

Pad with virtual 1s at both ends. `dp[i][j]` = max coins from bursting all balloons in the interval [i, j].

**Time O(n³), Space O(n²).**

This is the textbook example of "reframe the problem to pick a different decision point." Always worth practicing.


In [ ]:
def max_coins(nums):
    nums = [1] + nums + [1]
    n = len(nums)
    dp = [[0]*n for _ in range(n)]
    for length in range(2, n):                  # length = j - i, gap between boundaries
        for i in range(n - length):
            j = i + length
            for k in range(i+1, j):
                dp[i][j] = max(dp[i][j],
                               nums[i]*nums[k]*nums[j] + dp[i][k] + dp[k][j])
    return dp[0][n-1]

assert max_coins([3,1,5,8]) == 167
assert max_coins([1,5]) == 10
print("max_coins: OK")


### 15.3 Boolean Parenthesization

Count ways to parenthesize a boolean expression `T|F|T&F^...` to evaluate to True.

**Recurrence (sketch).** For each operator position k as the "last" operator, multiply the True/False counts of the left and right subexpressions and combine according to the operator's truth table.

Two states per interval: `T[i][j]` and `F[i][j]`. **Time O(n³).** Classic interval DP variant — know it exists.

### 15.4 Egg Drop (LC 887)

K eggs, N floors. Min worst-case drops to find the critical floor.

**Naive recurrence.**
```
dp[K][N] = 1 + min over f in [1, N] of  max(dp[K-1][f-1], dp[K][N-f])
                                              egg breaks    egg survives
```

O(K·N²). The O(K·N·logN) and O(K·N) optimizations exist via binary search on the inner loop or by inverting the question to "max floors solvable in M drops with K eggs."


In [ ]:
def super_egg_drop(k, n):
    # Inverted DP: dp[m][k] = max floors testable with m moves and k eggs. O(k·log n).
    dp = [[0]*(k+1) for _ in range(n+1)]
    m = 0
    while dp[m][k] < n:
        m += 1
        for kk in range(1, k+1):
            dp[m][kk] = dp[m-1][kk-1] + dp[m-1][kk] + 1
    return m

assert super_egg_drop(1, 2) == 2
assert super_egg_drop(2, 6) == 3
assert super_egg_drop(3, 14) == 4
print("super_egg_drop: OK — O(k·log n) via inverted DP")


### Interval DP practice problems

| Concept | LeetCode |
|---------|----------|
| Matrix Chain Multiplication | GFG classic |
| Burst Balloons | LC 312 |
| Strange Printer | LC 664 |
| Remove Boxes | LC 546 |
| Stone Game (variants) | LC 877, 1140, 1690 |
| Egg Drop | LC 887 |
| Minimum Cost to Cut a Stick | LC 1547 |
| Predict the Winner | LC 486 |


## 16. Stock problems — state machine DP

Six classic stock-trading problems (LC 121, 122, 123, 188, 309, 714). All of them are solved by the same **state machine DP** pattern, with slight variations.

**The state.** `dp[i][k][holding]`:
- `i` = day index
- `k` = transactions remaining (or completed, depending on convention)
- `holding` = 0 (cash) or 1 (own a share)

**Transitions:**
```
dp[i][k][0] = max(dp[i-1][k][0],  dp[i-1][k][1] + price[i])     # rest or sell
dp[i][k][1] = max(dp[i-1][k][1],  dp[i-1][k-1][0] - price[i])    # rest or buy
```

Selling decrements `k`'s available transactions (convention varies — let's say buying consumes the transaction credit).

Memorize this state machine; the variants just differ in what `k` is allowed and what additional constraints exist.

### 16.1 Best Time to Buy and Sell Stock — single transaction (LC 121)

Track min price seen so far and max profit at each day. Equivalent to: `max(prices[j] - min(prices[0..j]))` over all j. **O(n) time, O(1) space.**


In [ ]:
def max_profit_1(prices):
    min_price = float('inf')
    best = 0
    for p in prices:
        min_price = min(min_price, p)
        best = max(best, p - min_price)
    return best

assert max_profit_1([7,1,5,3,6,4]) == 5
assert max_profit_1([7,6,4,3,1]) == 0
print("max_profit_1: OK — O(n), O(1)")


### 16.2 Unlimited transactions (LC 122)

Greedy: sum all positive consecutive differences. Equivalent to capturing every upswing.

**Why the greedy works.** Any optimal profit can be decomposed into "buy on day i, sell on day j" for some i < j. The profit = sum of consecutive `prices[k+1] - prices[k]` for k in [i, j-1]. Negative consecutive diffs are pure loss — skip them. **O(n), O(1).**


In [ ]:
def max_profit_2(prices):
    return sum(max(0, prices[i] - prices[i-1]) for i in range(1, len(prices)))

assert max_profit_2([7,1,5,3,6,4]) == 7         # 1→5 (+4), 3→6 (+3) = 7
assert max_profit_2([1,2,3,4,5]) == 4
print("max_profit_2: OK")


### 16.3 At most 2 transactions (LC 123)

Track 4 states across the timeline: bought1, sold1, bought2, sold2. Each day update all 4 with the state machine logic. **O(n), O(1).**


In [ ]:
def max_profit_3(prices):
    b1 = b2 = float('-inf')         # cash after first buy / second buy (so we hold)
    s1 = s2 = 0                     # cash after first sell / second sell
    for p in prices:
        b1 = max(b1, -p)            # buy first (cash goes from 0 to -p)
        s1 = max(s1, b1 + p)        # sell first
        b2 = max(b2, s1 - p)        # buy second (from cash s1)
        s2 = max(s2, b2 + p)        # sell second
    return s2

assert max_profit_3([3,3,5,0,0,3,1,4]) == 6
assert max_profit_3([1,2,3,4,5]) == 4
print("max_profit_3: OK — 4-state machine, O(n), O(1)")


### 16.4 At most k transactions (LC 188)

Generalize to k transactions: dp arrays of size 2k. **O(n·k), O(k).** If k >= n/2, falls back to unlimited (case 16.2).


In [ ]:
def max_profit_k(k, prices):
    n = len(prices)
    if n == 0 or k == 0: return 0
    if k >= n // 2: return max_profit_2(prices)
    buys = [float('-inf')] * (k+1)
    sells = [0] * (k+1)
    for p in prices:
        for j in range(1, k+1):
            buys[j] = max(buys[j], sells[j-1] - p)
            sells[j] = max(sells[j], buys[j] + p)
    return sells[k]

assert max_profit_k(2, [3,2,6,5,0,3]) == 7
assert max_profit_k(2, [2,4,1]) == 2
print("max_profit_k: OK")


### 16.5 With cooldown (LC 309)

After selling, must wait one day before buying again. Add an extra state: `cooldown`.

**Transitions.**
```
hold[i]   = max(hold[i-1],  cash[i-1] - price[i])   # rest or buy from cash
cash[i]   = max(cash[i-1],  cool[i-1])              # rest or come out of cooldown
cool[i]   = hold[i-1] + price[i]                    # entered cooldown by selling
```
Final answer = max(cash, cool).


In [ ]:
def max_profit_cooldown(prices):
    if not prices: return 0
    hold = -prices[0]
    cash = 0
    cool = 0
    for p in prices[1:]:
        new_hold = max(hold, cash - p)
        new_cool = hold + p
        new_cash = max(cash, cool)
        hold, cash, cool = new_hold, new_cash, new_cool
    return max(cash, cool)

assert max_profit_cooldown([1,2,3,0,2]) == 3
print("max_profit_cooldown: OK")


### 16.6 With transaction fee (LC 714)

Subtract `fee` on every sell.


In [ ]:
def max_profit_fee(prices, fee):
    hold = -prices[0]
    cash = 0
    for p in prices[1:]:
        hold = max(hold, cash - p)
        cash = max(cash, hold + p - fee)
    return cash

assert max_profit_fee([1,3,2,8,4,9], 2) == 8
print("max_profit_fee: OK")


**The stock state-machine takeaway.** Six problems, one pattern. In an interview, **draw the state machine** before writing code. Show the states (hold/cash, possibly cooldown) and transitions. That visual narration is gold.


## 17. Advanced patterns (know these exist)

### 17.1 Bitmask DP

When state requires tracking a subset of items (and N ≤ 20-ish), encode the subset as an integer bitmask.

**Canonical problem:** Travelling Salesman Problem (TSP). State `dp[mask][i]` = min cost to visit cities in `mask` ending at city `i`. **O(2ⁿ · n²)** time.

**Other classics:**
- Partition to K Equal Sum Subsets (LC 698)
- Find the Shortest Superstring (LC 943)
- Smallest Sufficient Team (LC 1125)

Trigger: N ≤ 16-20, you need "any subset of N items." Brute force is 2ⁿ, and DP collapses the redundant orderings.

### 17.2 DP on trees

Root the tree. `dp[node]` = some value computed for the subtree at `node`. Compute bottom-up (post-order DFS).

**Canonical:**
- House Robber III (LC 337): `dp[node] = (rob_this, dont_rob_this)`.
- Binary Tree Maximum Path Sum (LC 124): track best downward path from each node.
- Diameter of Binary Tree (LC 543).
- Longest Path in a Tree.

State usually has 2-3 components per node.

### 17.3 Digit DP

Counting how many integers in [L, R] satisfy some digit-based predicate. State = `(position, tight_bound, started, ...)`. Used in competitive programming more than typical interviews — but appears occasionally (LC 233 Number of Digit One, LC 600 Non-negative Integers Without Consecutive Ones).

### 17.4 DP with bitmask + position (Profile DP)

Used in tile-laying problems (e.g., domino tilings of m×n grid). Mostly out of interview scope but mentioned for completeness.

### Advanced practice problems

| Concept | LeetCode |
|---------|----------|
| TSP (bitmask) | LC 943, LC 847 |
| Partition K Equal Sum Subsets | LC 698 |
| Smallest Sufficient Team | LC 1125 |
| House Robber III (tree DP) | LC 337 |
| Binary Tree Max Path Sum | LC 124 |
| Number of Digit One | LC 233 |
| Non-negative Integers Without Consecutive Ones | LC 600 |


## 18. Synthesis — cheat sheet, complexity table, narration, common bugs

### 18.1 Pattern-recognition cheat sheet

| Problem signal | Reach for |
|---|---|
| "Fibonacci-like": current depends on last 1-2 values | 1D linear DP, O(1) space |
| "Count ways to reach N" | 1D linear DP |
| "Decode / parse with multi-char options" | 1D DP looking back 1-2 chars |
| "Subarray min/max sum" | Kadane (O(n), O(1)) |
| "Max in 1D, but signs flip" | Track running max AND min (product subarray) |
| "Can we reach the end?" / "Min jumps?" | Greedy frontier DP |
| "Pick or skip with capacity" | 0/1 Knapsack — iterate capacity **reverse** in 1D |
| "Min/Max with unlimited reuse" | Unbounded knapsack — iterate capacity **forward** |
| "Count subsets / combinations summing to target" | Knapsack counting variant |
| "Combinations vs permutations" | Coin OUTER = combinations; amount OUTER = permutations |
| "Longest subsequence with property X" | LIS family — O(n²) DP or O(n log n) patience |
| "Two strings, align / transform" | 2D DP `dp[i][j]` (LCS, edit distance) |
| "Palindromic property in a substring" | 2D DP by interval length, or expand-around-center |
| "Grid, paths or sums, only right/down" | 2D grid DP, O(n) space rolling row |
| "Choose a split point on an interval" | Interval DP (MCM, burst balloons) — iterate by length |
| "Burst/remove order matters" | Reframe: pick the LAST element to burst |
| "Stock trading variants" | State machine: hold / cash / (cooldown) |
| "Subset of N ≤ 20 with constraints" | Bitmask DP |
| "Tree, aggregate over subtrees" | DP on trees (post-order DFS) |

### 18.2 Complexity reference table

| Problem | Time | Space | Optimization made |
|---------|------|-------|-------------------|
| Fibonacci (naive) | O(φⁿ) | O(n) | — |
| Fibonacci (memoized) | O(n) | O(n) | Cache overlapping subproblems |
| Fibonacci (tabulation) | O(n) | O(n) | Remove recursion overhead |
| Fibonacci (rolling) | O(n) | **O(1)** | Only last 2 values needed |
| Fibonacci (matrix exp) | **O(log n)** | O(log n) | Fast exponentiation |
| Climbing stairs / House Robber | O(n) | O(1) | Same as Fibonacci pattern |
| Decode Ways | O(n) | O(1) | Rolling pointer |
| Kadane's max subarray | O(n) | O(1) | Single-pass DP |
| Max product subarray | O(n) | O(1) | Track max AND min |
| Jump Game / Jump Game II | O(n) | O(1) | Greedy frontier |
| 0/1 Knapsack | O(N·W) | O(W) | 1D, reverse iteration |
| Subset Sum | O(N·target) | O(target) | 1D boolean DP |
| Coin Change (min coins) | O(amt·|coins|) | O(amt) | 1D unbounded |
| Coin Change II (count comb) | O(amt·|coins|) | O(amt) | Coins outer for combos |
| LIS (DP) | O(n²) | O(n) | — |
| LIS (patience sort) | **O(n log n)** | O(n) | Binary-search on tails |
| LCS / Edit Distance | O(m·n) | O(min(m,n)) | 2-row rolling |
| Longest Palindromic Substring | O(n²) | O(1) | Expand around center |
| Longest Palindromic Subsequence | O(n²) | O(n²) | Interval DP |
| Min Path Sum | O(m·n) | O(1) | In-place |
| Maximal Square | O(m·n) | O(n) | Rolling row |
| Dungeon Game | O(m·n) | O(m·n) | Backward DP |
| Matrix Chain Mult | O(n³) | O(n²) | Interval DP |
| Burst Balloons | O(n³) | O(n²) | Pick LAST to burst |
| Stock — 1 transaction | O(n) | O(1) | — |
| Stock — k transactions | O(n·k) | O(k) | State machine |
| TSP (bitmask DP) | O(2ⁿ·n²) | O(2ⁿ·n) | Subset encoding |
| Tower of Hanoi | O(2ⁿ) | O(n) | Lower bound — no DP possible |
| Josephus problem | O(n) | O(n) → O(1) iter | Index transform recurrence |
| Fast exponentiation | O(log n) | O(log n) → O(1) iter | Halve + square |
| Merge sort | O(n log n) | O(n) | D&C |
| Quicksort (avg/worst) | O(n log n) / O(n²) | O(log n) / O(n) | D&C with pivot |

### 18.3 Interview narration prep — verbatim answers

These are the questions interviewers will ask. Practice the answers out loud.

**Q: When does DP apply?**
> Two conditions: overlapping subproblems and optimal substructure. Overlapping subproblems means the same subproblem is solved more than once in the naive recursion — memoization fixes this. Optimal substructure means the optimal answer to the full problem can be built from optimal answers to subproblems. If only the first holds without the second, memoization speeds up the computation but the answer might still be wrong — you'd need a different formulation.

**Q: Memoization vs Tabulation — when to use which?**
> Memoization is top-down: I write the recursion that matches the recurrence directly, then add a cache. Tabulation is bottom-up: I fill a table from base cases. Both have the same asymptotic complexity. Memoization is easier to write because it follows the natural recursive thought. Tabulation has no recursion overhead and is easier to space-optimize, but you have to figure out the iteration order. In a sparse state space — where only a few subproblems are actually needed — memoization wins because it only computes what it needs. In a dense state space, tabulation is preferred. I usually start with memoization in an interview, then offer to convert if asked.

**Q: Why is 0/1 knapsack 1D iteration reverse but unbounded forward?**
> In 0/1 knapsack each item is used at most once. When we collapse the 2D dp to 1D, dp[w] refers to the previous row's value until we overwrite it. If we iterate forward, dp[w-wt] might already be updated with the current item — so taking it again means using the same item twice. Iterating reverse keeps dp[w-wt] referring to the previous row. For unbounded, we *want* the item reusable, so forward iteration is correct.

**Q: Why is patience-sort LIS O(n log n)?**
> `tails[k]` stores the smallest possible tail of an increasing subsequence of length k+1. It stays sorted. For each new element we binary search to find where it goes — O(log n) per element. We either extend the array (LIS got longer) or overwrite an existing tail (made future extensions easier). Total O(n log n). Important caveat: `tails` is not the actual LIS — only its length. To reconstruct the LIS, you need parent pointers.

**Q: Why does Kadane's algorithm work?**
> The key insight is that the maximum subarray ending at index i either extends the maximum subarray ending at i-1, or starts fresh at i. There's no third option. So dp[i] = max(nums[i], dp[i-1] + nums[i]). We only need dp[i-1], so O(1) space.

**Q: For Coin Change II, why does the order of loops matter?**
> If coins are the outer loop and amount is inner, we count combinations — once we move past a coin we never revisit it, so an order is enforced. If amount is outer and coins are inner, we count permutations — at each amount we consider every coin, allowing different orderings of the same multiset to be counted separately.

**Q: Why is the recursion for Burst Balloons "pick the last balloon" instead of "pick the first"?**
> If we pick the first balloon to burst in an interval, its neighbors change with every subsequent burst — making the subproblem state hard to define. If we pick the **last** balloon to burst, by definition all the others in the interval are already gone, so its neighbors are exactly the boundaries of the interval. That gives a clean recurrence: dp[i][j] = max over k of nums[i-1] * nums[k] * nums[j+1] + dp[i][k-1] + dp[k+1][j].

**Q: For Dungeon Game, why does forward DP fail?**
> The minimum HP needed at (i,j) depends on what's downstream, not what's upstream. If we fill forward, we have to track both current HP and minimum HP along the path simultaneously — and we'd be making the wrong greedy choice when paths split. Filling backward, dp[i][j] = "min HP needed entering this cell to survive" naturally encodes the future requirement.

**Q: How do you optimize 2D DP to 1D?**
> When dp[i][...] depends only on dp[i-1][...], we keep only the previous row. We allocate one row of size O(cols) and reuse it. Sometimes we need two rows (prev and curr); sometimes one is enough with careful iteration order. This drops space from O(m·n) to O(min(m,n)).

### 18.4 Common bugs and traps

1. **Forgetting the base case** — `RecursionError` on first call. Always state base cases explicitly.

2. **Wrong return contract in recursion** — e.g., returning min vs max, or returning the value vs the path. Decide what your function returns BEFORE writing it.

3. **Mutable default arguments** — `def f(memo={})` shares the dict across calls. Use `None` and initialize inside.

4. **0/1 knapsack iterating capacity forward** — silently solves unbounded knapsack instead. Items get reused.

5. **Coin Change II with swapped loops** — counts permutations instead of combinations.

6. **LIS strict vs non-strict** — `bisect_left` for strict increase, `bisect_right` for non-decreasing. Swap and you'll get wrong answers on duplicates.

7. **Edit distance off-by-one** — `dp[i][j]` refers to prefixes `s1[:i]` and `s2[:j]`, NOT to characters `s1[i]`, `s2[j]`. So you compare `s1[i-1]` to `s2[j-1]`.

8. **Palindrome DP iterating by i, j directly** — dependencies on `dp[i+1][j-1]` are not yet computed. Always iterate **by interval length**.

9. **Reading LIS from the tails array** — tails is not the actual LIS, just a length tracker. The "subsequence" you'd read off may not even be a valid subsequence of the original array.

10. **Burst Balloons "pick first"** — leads to a non-recursive state. Always pick the LAST balloon in the interval.

11. **Forgetting to handle the empty string / array** in DP base cases.

12. **Memoizing on mutable state** — e.g., passing a list as a parameter. Convert to tuple or use a different parameter that captures the relevant state.

13. **Stock state machine missing the cooldown 1-day rule** — buying immediately after selling. Need to explicitly route through the cooldown state.

14. **Mixing up dp[i][j] semantics** — "ending at index i,j" vs "for prefixes of length i,j". The off-by-one bugs come from inconsistency. Pick one and stick with it.

15. **Wrong recurrence direction in iteration** — Dungeon Game's backward DP, palindrome's length-outer, interval DP's length-outer. If your dp[i][j] depends on later values, you must iterate from larger to smaller (or by length).

16. **`@cache` on a method** — caches by `self` identity, which can cause subtle bugs. Prefer functions, or `@cache` carefully.

17. **Setting recursion limit too low** — for deep memoization, you need `sys.setrecursionlimit(10**6)`. Better still: convert to tabulation.

18. **Returning -1 vs inf vs None for "impossible"** — be consistent within a problem. Coin Change uses inf during fill and returns -1 at the end if dp[amount] is still inf.

### 18.5 Final summary table — the 8 master DP patterns

| # | Pattern | State | Time | Canonical problem |
|---|---------|-------|------|-------------------|
| 1 | 1D linear | dp[i] | O(n) | Climbing stairs, House Robber |
| 2 | Kadane / running | running max | O(n) | Max subarray, max product |
| 3 | 0/1 Knapsack | dp[i][w] | O(N·W) | Subset sum, Partition equal |
| 4 | Unbounded knapsack | dp[w] | O(W·k) | Coin change, Rod cutting |
| 5 | LIS family | dp[i] or tails | O(n²) or O(n log n) | LIS, Russian dolls |
| 6 | 2-string / 2D | dp[i][j] | O(m·n) | LCS, Edit distance |
| 7 | Grid | dp[i][j] | O(m·n) | Unique paths, Min path sum |
| 8 | Interval | dp[i][j] by length | O(n³) | MCM, Burst balloons |
| + | State machine | dp[i][state] | O(n·states) | Stock problems |
| + | Bitmask | dp[mask][i] | O(2ⁿ · n²) | TSP, K-partition |
| + | Tree DP | dp[node][state] | O(n) | House Robber III, Max Path Sum |

That's the whole topic. If you can recognize which pattern a problem belongs to within 60 seconds, write the recurrence within another 2-3 minutes, and walk the optimization ladder explicitly in your narration, you're in the top tier.
